# Обучение модели и получение масок

Загружаем недостающие библиотеки

In [1]:
!pip install segmentation_models

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 50 kB 5.9 MB/s 


Подключаем библиотеки

In [2]:
import os
from random import seed, shuffle
import cv2
import keras.backend as K
import segmentation_models as sm
import tensorflow as tf
from PIL import Image
from keras.callbacks import EarlyStopping
from numpy import array, zeros, expand_dims, uint8, ndarray
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from tqdm.keras import TqdmCallback
from keras.losses import binary_crossentropy
import gc
import glob
import math
from statistics import mean

Segmentation Models: using `keras` framework.


Задаём ширину и высоту изображений для обучения. 1536 пикселей, чтобы обойтись без сжатия до подачи на вход сети. 

Указываем количество цветовых каналов и общий seed для всех случайных величин

In [3]:
IMG_HEIGHT = 1536
IMG_WIDTH = 1536

IMG_CHANNELS = 3

SEED = 500

TRAINING_MODE = True

Инициализируем keras и случайные величины

In [4]:
sm.set_framework('tf.keras')
tf.keras.utils.set_random_seed(SEED)
seed(SEED)

Указываем корневую папку проекта на Google.Drive

In [5]:
root = '/content/drive/MyDrive/eye'

Подключаем Google Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Указываем в каких папках лежат обучающие маски, изображения для обучения и изображения для определения масок

In [7]:
masksPath = os.path.join(root, 'masks-2-pure')
trainPath = os.path.join(root, 'train')
checkPath = os.path.join(root, 'check')

Получение списка файлов из какой-либо папки, если их имена содержатся в списке validNames

In [8]:
def getFiles(trainData, resultData, validNames):
    if resultData is not None:
        masks = { filename: os.path.join(resultData, filename) for filename in os.listdir(resultData) if filename.endswith(".png") and (validNames is None or filename in validNames) }
    else:
        masks = None

    images = { filename: os.path.join(trainData, filename) for filename in os.listdir(trainData) if filename.endswith(".png") and (validNames is None or filename in validNames) }

    return images, masks

Превращаем прямоугольные изображения и маски в квадратные, отбросив либо поровну слева и справа, либо, для некоторых масок и изображений только слева или только справа.
Маска 206 имеет неправильный размер, её просто отбрасываем.

In [9]:
left  = ["141.png", "332.png", "455.png", "657.png", "1028.png", "1042.png", "939.png"]
right = ["197.png"]
exclude = ["206.png"]

offsetX = 1624 - 1536
offsetY = 1536 - 1232

Дополнение изображения по высоте до нужного размера. Пустые области заполняются чёрным цветом

In [10]:
def expandImageHeight(image, height):
    half = (height - image.shape[0]) // 2
    image_extended = ndarray((height,) + image.shape[1:], dtype=image.dtype)

    image_extended[half:, :] = 0
    image_extended[half:image.shape[0] + half, :] = image
    image_extended[image.shape[0] + half:, :] = 0

    return image_extended

Обрезка изображения по горизонтали и дополнение по вертикали

In [11]:
def cropAndResize(image, filename):
    y = 0
    h = 1232
    w = 1536

    if filename in exclude:
        return image
    elif filename in left:
        x = 0
    elif filename in right:
        x = offsetX
    else:
        x = offsetX // 2

    crop = image[y:y + h, x:x + w]

    extended = expandImageHeight(crop, 1536)

    return extended

Обратное преобразование изображения: по вертикали убираются добавленные ранее области, по горизонтали изображение дополняется чёрным цветом до нужной ширины

In [12]:
def restoreSize(image, filename):
    black = zeros((1232, 1624), dtype="uint8")

    if filename in exclude:
        return image
    if filename in left:
        start = 0
    elif filename in right:
        start = offsetX
    else:
        start = offsetX // 2

    x = 0
    y = offsetY // 2
    h = 1232
    w = 1536

    crop = image[y:y + h, x:x + w]

    black[0:h, start:(w + start)] = crop

    return black

Создать набор изображений и масок для обучения или для распознавания из списка файлов. Если для распознавания, то masksList = None. Можно указать набор функций трансформации Albumentations. А также, добавлять эти изображения, начиная с любого заданного индекса в итоговом массиве. Чтобы объединять блоки изображений

In [13]:
def createSimpleDataset(startIndex, imagesList, masksList, images, masks, transforms):
    for idImage, (filename, fullName) in enumerate(tqdm(imagesList.items())):
        if masksList is not None:
            mask = cv2.imread(masksList[filename])
            mask = mask[:, :, 0]
            mask = cropAndResize(mask, filename)
            mask = expand_dims(mask, axis=-1)
        else:
            mask = None

        image = cv2.imread(fullName)
        image = cropAndResize(image, filename)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


        if transforms is not None:
            augmented = transforms(image = array(image), mask = mask)
            del image
            image = augmented["image"]
            if mask is not None:
                del mask
                mask = augmented["mask"]

        if mask is not None:
            masks[startIndex + idImage] = mask

        images[startIndex + idImage] = image

Создать набор изображений и масок для обучения или предсказания. Масок может не быть, если это предсказание. Можно указать какой процент изображений отбросить случайным образом (ради экономии памяти видеокарты). Можно указать функции трансформации Albumentations и какой процент от основного массива изображений должен быть преобразован и добавлен в обучение.

In [14]:
def createDataset(imagesList, masksList, transformsBasic, basicSkipProportion, transformsMore, moreSkipProportion):
    basicLength = len(imagesList)
    basicSize = 0 if basicSkipProportion == 1 else (basicLength if basicSkipProportion == 0 else int(float(basicLength) * (1 - basicSkipProportion)))
    moreSize = 0 if moreSkipProportion == 1 else (int(float(basicLength) * (1 - moreSkipProportion)) if transformsMore is not None and 0 <= moreSkipProportion < 1 else 0)
    images = zeros((basicSize + moreSize, IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS), dtype = uint8)

    if masksList is not None:
        masks = zeros((basicSize + moreSize, IMG_HEIGHT, IMG_WIDTH, 1), dtype = bool)
    else:
        masks = None

    imagesNames = list(imagesList.keys())

    if basicSize > 0:
        shuffle(imagesNames)
        basicImagesList = {filename: fullName for filename, fullName in imagesList.items() if filename in imagesNames[:basicSize]}
        createSimpleDataset(startIndex = 0, imagesList = basicImagesList, masksList = masksList, images = images, masks = masks, transforms = transformsBasic)

    if moreSize > 0:
        shuffle(imagesNames)
        moreImagesList = {filename: fullName for filename, fullName in imagesList.items() if filename in imagesNames[:moreSize] }
        createSimpleDataset(startIndex = basicSize, imagesList = moreImagesList, masksList=masksList, images=images, masks=masks, transforms = transformsMore)

    return images, masks

Функции трансформации изображений основного и дополнительного набора.

Сделано, но не использовалось, так как по-первых нехватало видеопамяти на Google.Colab, а во-вторых, аугментация улучшала модель, но ухудшала score, так как задача найти не сосуды, а сосуды, отмеченные студентами.

In [15]:
transformsBasic = None
transformsMore = None

Подготавливаем список файлов из папки масок. Разбиваем его на файлы для обучения и для проверки обучения.

Затем получаем список изображений для обучения и контроля так, чтобы имена соответствовали соответствующему набору масок.

In [16]:
validFilenames = [ filename for filename in os.listdir(masksPath) if filename.endswith("png") and not filename.startswith(".") ]

validTrain, validTest = train_test_split(validFilenames, test_size = 0.1, random_state = SEED)

imagesFilesTrain, maskFilesTrain = getFiles(trainPath, masksPath, validTrain)
imagesFilesTest,  maskFilesTest  = getFiles(trainPath, masksPath, validTest)

Если включён режим обучения (может быть ещё и режим просто формирования картинок от сохранённой модели, без обучения), то формируем набор данных для обучения и тренировки.

Для обучения отбрасываем 25% картинок. Причина - нехватка видеопамяти Google.Colab. Если бы хватало, отбрасывать не пришлось.

In [17]:
if TRAINING_MODE:
    X_train, y_train = createDataset(imagesList = imagesFilesTrain, masksList = maskFilesTrain,
                                     transformsBasic = transformsBasic, basicSkipProportion = 0.25,
                                     transformsMore = transformsMore, moreSkipProportion = 1)
    
    X_test,  y_test  = createDataset(imagesList = imagesFilesTest,  masksList = maskFilesTest,
                                     transformsBasic = None, basicSkipProportion = 0,
                                     transformsMore = None, moreSkipProportion = 1)

100%|██████████| 57/57 [00:47<00:00,  1.20it/s]


Функция Dice-Loss потери, основанная на совпадении площади предсказанной и обучающей маски.

Сделано канонически по формуле.

Чтобы избежать деления на 0 в ситуации, когда маска пустая и сеть не нашла нужные участки, добавлен небольшой поправочный коэффициент в числитель и знаменатель (в этом случае diceLoss будет 1, что соответствует ожиданиям)

In [18]:
def diceLoss(targetsPure, inputsPure):
    targets = tf.cast(K.flatten(targetsPure), tf.float32)
    inputs = tf.cast(K.flatten(inputsPure), tf.float32)

    intersection = K.sum(targets * inputs)
    dice = (2 * intersection + 1e-6) / (K.sum(targets) + K.sum(inputs) + 1e-6)
    return 1 - dice

Функция, суммирующая бинарную кросс-энтропию и Dice-Loss. Ведь нам надо минимизировать не только разницу по площади, но и одинаковость ошибок разметки

In [19]:
def bceDiceLoss(y_true, y_pred):
    return K.mean(binary_crossentropy(y_true, y_pred)) + diceLoss(y_true, y_pred)

Устанавливаем параметры обучения.

Сеть efficientnetb0 давала наилучшие результаты. Простестированы было несколько разных сетей.
Обучение - как обычно, Adam,
Функция потерь,
Обучение/загрузка модели/дообучение загруженной модели
Количество блоков в обучении (тут максимум 1 и спасибо, что хоть так хватило памяти)
Количество эпох. В примере - 5, но это потому, что colab совсем не даёт больше времени на использование видеокарты, а colab+ больше не купить. 

В модели с чемпионата было 30 эпох.

Функция остановки обучения, если 10 раз подряд не было улучшений в тестирующей выборке. И восстановление шага, когда был лучший результат по тестирующей выборке.

In [20]:
settings = \
{
    'model': sm.Unet('efficientnetb0', classes = 1, activation = 'sigmoid'),
    "optimizer": tf.keras.optimizers.Adam(learning_rate = 0.001),
    "loss": bceDiceLoss,
    "metrics": ["accuracy", sm.metrics.iou_score],
    'train': TRAINING_MODE,
    "continue-train": False,
    "saved-model": "models-entropy/unet-efficientnetb0.ckpt",
    "batch-size": 1,
    "epochs": 5,
    "callbacks": [EarlyStopping(monitor = 'val_iou_score', mode = 'max', patience = 10, verbose = 0, restore_best_weights = True)]
}

16818176/16804768 [==============================] - 0s 0us/step


Обучение модели, либо загрузка и дообучение, с указанными параметрами,
либо просто загрузка

In [21]:
model = settings['model']

if settings['train']:
    if settings["continue-train"]:
        model.load_weights(os.path.join(root, settings["saved-model"])).expect_partial()

    model.compile(settings["optimizer"], settings["loss"], settings["metrics"])

    gc.collect()

    model.fit(x=X_train,y=y_train, batch_size=settings["batch-size"], epochs=settings["epochs"], validation_data = (X_test, y_test), verbose = 0,
              shuffle = True, callbacks = settings["callbacks"] + [TqdmCallback(verbose = 2)])
    
    del X_train
    del y_train

    gc.collect()

    model.save_weights(os.path.join(root, settings["saved-model"]))

else:
    model.load_weights(os.path.join(root, settings["saved-model"])).expect_partial()

0epoch [00:00, ?epoch/s]

  0%|          | 0.00/378 [00:00<?, ?batch/s]

  0%|          | 0.00/378 [00:00<?, ?batch/s]

  0%|          | 0.00/378 [00:00<?, ?batch/s]

  0%|          | 0.00/378 [00:00<?, ?batch/s]

  0%|          | 0.00/378 [00:00<?, ?batch/s]

После обучения, создаём набор данных для определения масок.

In [22]:
imagesCheck, masksEmpty = getFiles(checkPath, None, None)

X_test, empty = createDataset(imagesList = imagesCheck, masksList = None,
                              transformsBasic = None, basicSkipProportion = 0,
                              transformsMore = None, moreSkipProportion = 1)

100%|██████████| 301/301 [00:26<00:00, 11.50it/s]


Выполняем определение масок.

После чего возвращаем их к исходному размеру.

In [24]:
for imageIndex in tqdm(range(len(imagesCheck))):
    imageName = list(imagesCheck.keys())[imageIndex]

    image = X_test[imageIndex]

    predictedMask = (model.predict(expand_dims(X_test[imageIndex], axis=0))[0].squeeze() * 255).astype(uint8)

    predictedImage = Image.fromarray(restoreSize(predictedMask, imageName))
    predictedImage = predictedImage.point(lambda pixel: 255 if pixel > 200 else 0)

    predictedImage.save(os.path.join(root, "result/" + imageName))

100%|██████████| 301/301 [02:30<00:00,  2.00it/s]


# Постобработка масок

Указываем папки для постобработки.
Исходная папка - result, где только что сформировались маски сетью
Папка с картинками для проверки цвета
Результирующая папка

In [25]:
resultPath = os.path.join(root, "prepared-result")
sourcePath = os.path.join(root, "result")
checkPath  = os.path.join(root, "check")

Устанавливаем режим - постобработки, либо поиска среднего цвета сосуда
(внезапно, не красный. Если анализировать цветовые каналы, сосуды не красные, это лишь обман зрения, результат коррекции изображения нейросетью в нашем мозгу. Красный весь глаз, а сосуды более серые на его фоне).

Также указываем порог разницы цвета, при которой считаем, что сосуд врядли бы привлёк внимание студента.

In [26]:
prepareMode = False

meanRed   = 105.98
meanGreen = 118.57
meanBlue  = 136.04

threshold = 70

Переменные для суммирования изменений, чтобы потом посмотреть итоги постобработки

In [27]:
allContours = []
removeContours = []

allContoursSquare = 0
removeContoursSquare = 0
addContoursSquare = 0

Постобработка маски.

Обводим все найденные сосуды контурами CV2. Маленькие вложенные контуры не разметит ни один уважающий себя студент, поэтому их закрашиваем белым. 

Маленькие контуры считаем либо погрешностью сети, либо не достойными внимания студента и отбрасываем, закрашивая чёрным.

Средние контуры проверяем на средний цвет. Если он ниже заданного порога, то студента они не привлекут. Их тоже отбрасываем.

In [28]:
def contoursChanged(image, mask) -> bool:
    global allContoursSquare
    global removeContoursSquare
    global addContoursSquare

    contours, hierarchy = cv2.findContours(mask[:,:,0], cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    if contours is None or hierarchy is None:
        return mask

    for contour, treeData in zip(contours, hierarchy[0,:,:]):
        area = cv2.contourArea(contour)

        # если это вложенный контур, 4-й элемент массива информации будет отличен от -1
        if treeData[3] != -1:
            if not prepareMode and area < 60:
                addContoursSquare += area

                cv2.fillPoly(mask, pts=[contour], color=(255, 255, 255))
            continue

        if prepareMode:
            collectMeanColor(image, contour)

        else:
            allContours.append(contour)

            allContoursSquare += area

            if area < 40 or (area < 245 and checkContourDifference(image, contour)):
                removeContours.append(contour)
                removeContoursSquare += area

                cv2.fillPoly(mask, pts=[contour], color=(0, 0, 0))

    return mask

Сбор среднего значения по цветовым каналам

In [29]:
redCollect = []
greenCollect = []
blueCollect = []

In [30]:
def collectMeanColor(image, contour):
    mask = zeros(image.shape, uint8)
    cv2.drawContours(mask, contour, -1, 255, -1)
    (red, green, blue, alpha) = cv2.mean(image, mask=mask[:,:,0])
    redCollect.append(red)
    greenCollect.append(green)
    blueCollect.append(blue)

Подсчёт разницы среднего цвета контура и среднего цвета всех контуров. Считается геометрическое расстояние по шкале RGB.

In [31]:
def checkContourDifference(image, contour) -> bool:
    mask = zeros(image.shape, uint8)
    cv2.drawContours(mask, contour, -1, 255, -1)
    (red, green, blue, alpha) = cv2.mean(image, mask=mask[:, :, 0])

    return math.sqrt((meanRed - red) ** 2 + (meanGreen - green) ** 2 + (meanBlue - blue) ** 2) > threshold

Выполняем постобработку для всех масок из исходной папки

In [32]:
for filename in tqdm(glob.glob(sourcePath + "/*.png")):
    pureName = filename.replace("\\", "/").split("/")[-1]

    mask = cv2.imread(filename)
    image = cv2.imread(os.path.join(checkPath, pureName))

    cv2.imwrite(os.path.join(resultPath, pureName), contoursChanged(image, mask),  [cv2.IMWRITE_PNG_COMPRESSION, 9])

100%|██████████| 301/301 [03:20<00:00,  1.50it/s]


Вывод результатов постобработки, либо средние значения по цветам

In [33]:
if prepareMode:
    print("Red: ", mean(redCollect), "Green:", mean(greenCollect), "Blue:", mean(blueCollect))

else:
    print("Removed: ", round(100.0 * len(removeContours) / len(allContours), 2), "%")
    print("Added: ", round(100.0 * addContoursSquare / allContoursSquare, 2), "%")
    print("Square: ", round(100.0 * removeContoursSquare / allContoursSquare, 2), "%")

Removed:  82.04 %
Added:  0.48 %
Square:  2.7 %
